# HAKE-MER — Step~0 cardinality (DistilBERT PLM baseline)

**PFE — Mariem El Wedani** · SIIVA · Encadrant académique : Sahbi Bahroun · 2025–2026


Fills the **Step~0** column of `tab:expe-cardinality` without changing canonical campaign F1 scores.

1. Ensures three `runs/distilbert_base_uncased_seed*_baseline_plm/best_model.pt` exist (trains missing seeds with the usual Step~0 protocol).
2. Runs `./run_step0_cardinality.sh` (test-only `by_gold_cardinality`).
3. Download zip → integrate into `step_eval_supplements_distilbert/`.

Reference macro: **Step~0 F1-macro test ≈ 0.486 ± 0.005** (canonical campaign JSON).

In [ ]:
import torch
if not torch.cuda.is_available():
    raise RuntimeError("Runtime → Change runtime type → GPU")
print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
import subprocess
from getpass import getpass
from pathlib import Path

import os
REPO = os.environ.get("GITHUB_REPO", "khalef-khalil/marii")
WORKDIR = Path("/content/marii")
PUBLIC_URL = f"https://github.com/{REPO}.git"
TRAIN_FLAGS = "--epochs 4 --batch-size 16 --lr 5e-5 --early-stopping-patience 0 --device cuda"

def clone_repo() -> None:
    if WORKDIR.is_dir():
        subprocess.run(["git", "-C", str(WORKDIR), "fetch", "origin", "main"], check=True)
        subprocess.run(["git", "-C", str(WORKDIR), "reset", "--hard", "origin/main"], check=True)
        return
    r = subprocess.run(["git", "clone", "--depth", "1", PUBLIC_URL, str(WORKDIR)], capture_output=True)
    if r.returncode == 0:
        return
    try:
        from google.colab import userdata
        token = userdata.get("GITHUB_TOKEN")
    except Exception:
        token = getpass("GitHub token: ")
    subprocess.run(["git", "clone", "--depth", "1", f"https://{token}@github.com/{REPO}.git", str(WORKDIR)], check=True)

clone_repo()
%cd {WORKDIR}
!git rev-parse --short HEAD

In [ ]:
!pip install -q -r requirements-train.txt

In [ ]:
from pathlib import Path

ck = sorted(Path("runs").glob("distilbert_base_uncased_seed*_baseline_plm/best_model.pt"))
print(len(ck), "/ 3 Step~0 checkpoints")
if len(ck) < 3:
    print("Training Step~0 PLM (missing seeds only in campaign runner)…")
    get_ipython().system(f"./run_baseline_campaign.sh --backbone distilbert-base-uncased {TRAIN_FLAGS}")

In [ ]:
!chmod +x run_step0_cardinality.sh && ALLOW_EVAL=1 DEVICE=cuda ./run_step0_cardinality.sh

In [ ]:
from pathlib import Path
print(Path("reference/artifacts/cardinality_snippet.tex").read_text())

In [ ]:
import zipfile
from google.colab import files
from pathlib import Path

out_name = "step0_cardinality_distilbert.zip"
zip_path = Path(f"/content/{out_name}")
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    bc = Path("reference/artifacts/step_eval_supplements_distilbert/campaigns/baseline_plm_distilbert_base_uncased_campaign.json")
    zf.write(bc, bc.name)
    zf.write(Path("reference/artifacts/cardinality_snippet.tex"), "cardinality_snippet.tex")
    for m in sorted(Path("runs").glob("distilbert_base_uncased_seed*_baseline_plm/metrics.json")):
        zf.write(m, f"metrics/{m.parent.name}/{m.name}")
files.download(str(zip_path))

Télécharger le zip et le notebook exécuté ; les archiver avec les autres campagnes (dossier `journaux_colab/` du livrable).